In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyodbc
import sys
import os
import time
from datetime import date
import re
import openpyxl
from openpyxl import Workbook, load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
import warnings as wr
wr.filterwarnings('ignore')

In [23]:
rc1 = pd.read_excel('RC1237_clean.xlsx',sheet_name='RC1')
rc1[rc1['TNSSN'].isin([16,17,18,19])]

,RC,ST,YR,SESON,NOVP,NOVCOD,STAT,DIST,STRA,VILL,...,DCKC,HSSN,GEOA,TCROPA,TNSSN,TGEOASN,RC61,RC62,RC63,REJ
13,1,1,2025,2,260,260,33,5,1,3,...,1.0,326,194.150,1.8600,19,3.4200,9,9,9,0
17,1,1,2025,2,260,260,33,5,3,1,...,1.0,761,2110.850,2.6150,19,4.6490,9,9,4,0
18,1,1,2025,2,260,260,33,5,3,2,...,1.0,2573,1061.760,1.4000,19,6.5300,9,9,9,0
34,1,1,2025,2,260,260,33,6,5,2,...,1.0,1442,159.340,0.8200,19,1.6030,9,9,9,0
56,1,1,2025,2,260,260,33,8,5,1,...,1.0,62,26.480,2.8700,17,4.2100,9,9,9,0
82,1,1,2025,2,260,260,33,12,2,1,...,1.0,3962,921.930,0.1000,17,1.7474,9,9,4,0
241,1,1,2025,2,260,260,33,34,4,1,...,0.0,1718,2123.350,0.8350,18,10.6020,4,9,9,0
268,1,1,2025,3,260,260,33,3,1,2,...,1.0,2834,139.860,0.0000,19,0.7040,4,4,4,0
317,1,1,2025,3,260,260,33,8,5,1,...,1.0,62,26.480,0.5200,17,4.2100,9,9,9,0
343,1,1,2025,3,260,260,33,12,2,1,...,1.0,3962,921.930,0.0000,17,1.7474,4,9,9,0


In [ ]:
def annex01(rc1):
    years = list(rc1['YR'].unique())
    states = list(rc1['STAT'].unique())
    seasons = list(rc1['SESON'].unique())
    samples = list(rc1['ST'].unique())

    df_annex01 = pd.DataFrame()
    
    for year in years:
        for state in states:
            for season in seasons:
                for sample in samples:
                    df_annex01['year'] = year
                    df_annex01['state'] = state
                    df_annex01['season'] = season
                    df_annex01['sample'] = sample

                    rc1_filtered = rc1[(rc1['YR']==year) & (rc1['STATE']==state) & (rc1['SESON']==season) & (rc1['ST']==sample)]
                    #planning
                    plan = rc1_filtered['NOVP'].mean()
                    df_annex01['novp'] = plan
                    
                    #received
                    nov_received = rc1_filtered.shape[0]
                    df_annex01['received'] = nov_received
                    df_annex01['received%'] = 100 * np.divide(nov_received, plan)

                    #received before COD
                    novcod = rc1_filtered['NOVCOD'].mean()
                    df_annex01['novcod'] = novcod
                    df_annex01['novcod%'] = 100 * np.divide(novcod, plan)

                    #unrejected schedules df
                    rc1_unrej = rc1_filtered[rc1_filtered['REJ'] == 0]

                    #Analysed with a & b entries together
                    a_b_together = rc1_unrej[rc1_unrej['EPC']==1].shape[0]
                    df_annex01['a_b_together'] = a_b_together
                    df_annex01['a_b_together%'] = 100 * np.divide(a_b_together, plan)

                    #Analysed with a & b entries separately
                    a_b_separately = rc1_unrej[rc1_unrej['EPC']==2].shape[0]
                    df_annex01['a_b_separately'] = a_b_separately
                    df_annex01['a_b_separately%'] = 100 * np.divide(a_b_separately, plan)

                    #Analysed with a entries only
                    a_only = rc1_unrej[rc1_unrej['EPC']==3].shape[0]
                    df_annex01['a_only'] = a_only
                    df_annex01['a_only%'] = 100 * np.divide(a_only, plan)

                    #Analysed Total
                    analysed_total = np.sum([a_b_together, a_b_separately, a_only])
                    df_annex01['analysed_total'] = analysed_total
                    df_annex01['analysed_total%'] = 100 * np.divide(analysed_total, plan)

    

In [8]:
if __name__ == '__main__':

    rc1 = pd.read_excel('RC1237_clean.xlsx', sheet_name='RC1')
    rc2 = pd.read_excel('RC1237_clean.xlsx', sheet_name='RC2')
    rc3 = pd.read_excel('RC1237_clean.xlsx', sheet_name='RC3')
    rc4 = pd.read_excel('RC1237_clean.xlsx', sheet_name='RC7')
    
    df_annex01 = annex01(rc1)
    df_annex02 = annex01()
    df_annex03 = annex01()
    df_annex04 = annex01()

,RC,ST,YR,SESON,NOVP,NOVCOD,STAT,DIST,STRA,VILL,...,DCKC,HSSN,GEOA,TCROPA,TNSSN,TGEOASN,RC61,RC62,RC63,REJ
0,1,1,2025,2,260,260,33,1,1,1,...,1.0,990,499.520,0.195,20,4.0610,9,9,9,0
1,1,1,2025,2,260,260,33,1,1,2,...,1.0,2465,706.520,5.800,20,5.8000,9,9,9,0
2,1,1,2025,2,260,260,33,1,1,3,...,1.0,1333,295.840,0.970,20,1.6610,4,9,9,0
3,1,1,2025,2,260,260,33,1,2,1,...,1.0,2786,212.920,0.377,20,1.2550,9,9,9,0
4,1,1,2025,2,260,260,33,1,2,2,...,1.0,4599,749.350,1.730,20,2.8110,9,9,9,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
990,1,2,2025,3,260,227,33,38,1,5,...,1.0,1107,275.840,2.740,20,5.1750,9,9,4,0
991,1,2,2025,3,260,227,33,38,1,6,...,1.0,2562,445.680,4.255,20,5.0850,9,9,4,0
992,1,2,2025,3,260,227,33,38,2,3,...,1.0,0,130.830,0.000,20,2.9668,4,4,4,0
993,1,2,2025,3,260,227,33,38,3,3,...,1.0,799,164.805,0.000,20,5.8005,4,4,4,0
